# Examine search trace — BL family (v1)

Step-by-step inspection of ONE **best-first (BL) frontier**
search method on ONE question — for debugging and understanding
how the search tree is generated, not for measuring accuracy.
Nothing is written to `results/` or W&B.

This is the `mcts_bl_*`-focused sibling of
`examine_search_trace_v1.ipynb`. All eight BL variants share one
shape: they maintain an explicit global leaf frontier and, each
iteration, generate → expand `current`, add its children to the
frontier, then select the next node globally across the whole
frontier. What differs across the eight is only the frontier
*score*:

- `mcts_bl_cnt`  — PUCT (v01) / selectable path-aware score (v02),
- `mcts_bl_kube` — fractional-KUBE density,
- `mcts_bl_kdepth` — depth-discounted KUBE density,
- `mcts_bl_sem`  — diversity-adjusted value
  `ds_beta*q + ds_alpha*sched*sqrt(x^T V^-1 x)` (v02 adds a
  selectable value term via `score_mode`).

The v02 cores additionally fold terminals eagerly (cnt: into the
backprop; sem: into the diversity covariance V) — watch that in
the trace and in the tree dump (section 7).

Workflow:

1. pick `METHOD` + config `OVERRIDES` (section 2),
2. load the models (section 3), pick the question (section 4)
   and the trace `VERBOSITY` (section 5),
3. run the search (section 6) — the live trace streams into
   the cell output,
4. inspect the run post-hoc: the search tree, summary stats,
   completed nodes, per-step PRM re-score (sections 7–10).

How it works: the live trace reuses the `logging.fatal` /
`logging.error` calls already inside the search cores (the
launchers silence them; this notebook re-enables the root
logger). The post-hoc tree view works because the notebook
builds the `MCTS` agent itself and keeps it after
`mcts_search` returns — no monkey-patching needed.

GPU required (vLLM engine + PRM).

## 1. Setup

In [ ]:
import os

os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
sys.path.insert(0, "..")

import time
import random
import logging
import importlib

import numpy as np
import torch
from vllm import LLM
from hydra import initialize, compose
from hydra.core.config_store import ConfigStore
from omegaconf import OmegaConf

from core.reward_models import build_prm
from utils.configs import (
    ExpConfig,
    BLMCTSCntConfig, BLMCTSCntV02Config,
    BLMCTSKubeV01Config, BLMCTSKubeV02Config,
    BLMCTSKdepthV01Config, BLMCTSKdepthV02Config,
    BLMCTSSemConfig, BLMCTSSemV02Config,
)
from utils.load_data import load_data_hf
from notebook_utils import gpu_mem_used_gb, print_step_scores

assert torch.cuda.is_available(), "CUDA is required"
print(torch.cuda.get_device_name(0))

## 2. Pick the method and compose the config

`METHOD` selects the search core + its Hydra root config;
`OVERRIDES` is the same override list a launcher command would
take. Keep the budget tiny — the point is to watch the
mechanics, not to finish a real run.

Switching `METHOD` only re-imports the core module; the loaded
LLM/PRM are reusable as long as the `llm=`/`prm=` groups are
unchanged (otherwise re-run section 3, which needs a kernel
restart to free GPU memory first).

**Tracing the v02 score_mode arms:** `mcts_bl_cnt_v02` and
`mcts_bl_sem_v02` both take a `search.score_mode` knob. Add e.g.
`"search.score_mode=parent_blend"` (or `path_decay`) to
`OVERRIDES` to watch that value term reshape which frontier node
wins each selection. bl_sem_v02's default is `own` (= v01
behavior); bl_cnt_v02's default is `parent_blend`.

In [ ]:
# One entry per BL search variant. "embeds" marks the extra
# llm_vllm_embeds argument that only the sem family takes; the
# cnt/kube/kdepth mcts_search signatures omit it.
METHODS = {
    "mcts_bl_cnt_v01": {
        "module": "core.mcts_bl_cnt_search_v01_00_00",
        "config_name": "mcts_bl_cnt_v01_prm800k",
        "embeds": False,
    },
    "mcts_bl_cnt_v02": {
        "module": "core.mcts_bl_cnt_search_v02_00_00",
        "config_name": "mcts_bl_cnt_v02_prm800k",
        "embeds": False,
    },
    "mcts_bl_kube_v01": {
        "module": "core.mcts_bl_kube_search_v01_00_00",
        "config_name": "mcts_bl_kube_v01_prm800k",
        "embeds": False,
    },
    "mcts_bl_kube_v02": {
        "module": "core.mcts_bl_kube_search_v02_00_00",
        "config_name": "mcts_bl_kube_v02_prm800k",
        "embeds": False,
    },
    "mcts_bl_kdepth_v01": {
        "module": "core.mcts_bl_kdepth_search_v01_00_00",
        "config_name": "mcts_bl_kdepth_v01_prm800k",
        "embeds": False,
    },
    "mcts_bl_kdepth_v02": {
        "module": "core.mcts_bl_kdepth_search_v02_00_00",
        "config_name": "mcts_bl_kdepth_v02_prm800k",
        "embeds": False,
    },
    "mcts_bl_sem_v01": {
        "module": "core.mcts_bl_sem_search_v01_00_00",
        "config_name": "mcts_bl_sem_v01_prm800k",
        "embeds": True,
    },
    "mcts_bl_sem_v02": {
        "module": "core.mcts_bl_sem_search_v02_00_00",
        "config_name": "mcts_bl_sem_v02_prm800k",
        "embeds": True,
    },
}
METHOD = "mcts_bl_sem_v02"

# Recorded-experiments default budget. For the v02 score_mode
# families, append e.g. "search.score_mode=parent_blend" here.
OVERRIDES = [
    "llm=llama_1b",
    "search.gen_budget=80",
]

# Register the structured schemas (all BL variants — harmless),
# same as the launchers do, so the YAML binds onto typed
# dataclasses. Names match generate_mcts_bl_cnt.py /
# generate_mcts_sem.py exactly.
cs = ConfigStore.instance()
cs.store(name="exp_schema", node=ExpConfig)
cs.store(
    group="search", name="mcts_bl_cnt_v01_schema",
    node=BLMCTSCntConfig,
)
cs.store(
    group="search", name="mcts_bl_cnt_v02_schema",
    node=BLMCTSCntV02Config,
)
cs.store(
    group="search", name="mcts_bl_kube_v01_schema",
    node=BLMCTSKubeV01Config,
)
cs.store(
    group="search", name="mcts_bl_kube_v02_schema",
    node=BLMCTSKubeV02Config,
)
cs.store(
    group="search", name="mcts_bl_kdepth_v01_schema",
    node=BLMCTSKdepthV01Config,
)
cs.store(
    group="search", name="mcts_bl_kdepth_v02_schema",
    node=BLMCTSKdepthV02Config,
)
cs.store(
    group="search", name="mcts_bl_sem_v01_schema",
    node=BLMCTSSemConfig,
)
cs.store(
    group="search", name="mcts_bl_sem_v02_schema",
    node=BLMCTSSemV02Config,
)

spec = METHODS[METHOD]
core_mod = importlib.import_module(spec["module"])

with initialize(version_base=None, config_path="../conf"):
    cfg = compose(config_name=spec["config_name"], overrides=OVERRIDES)

print(f"method = {METHOD}  ({spec['module']})")
print(OmegaConf.to_yaml(cfg.search))

## 3. Load the models

Mirrors the launcher exactly (incl. `quantization` /
`load_format` / `seed`). The second pooling engine is built only
for the sem-v01 policy-embeds path; sem-v02 (and every
cnt/kube/kdepth variant) pulls embeds from the PRM forward pass
or needs none, and receives `None`.

In [ ]:
llm_vllm = LLM(
    model=cfg.llm.llm_dir,
    tensor_parallel_size=cfg.llm.tensor_parallel_size,
    max_model_len=cfg.llm.max_model_len,
    gpu_memory_utilization=cfg.llm.gpu_memory_utilization,
    enforce_eager=cfg.llm.enforce_eager,
    distributed_executor_backend=None,
    dtype=cfg.llm.dtype,
    quantization=cfg.llm.quantization,
    load_format=cfg.llm.load_format,
    seed=cfg.gen.seed,
)

llm_vllm_embeds = None
if spec["embeds"] and cfg.search.embeds_source == "policy":
    llm_vllm_embeds = LLM(
        model=cfg.llm.llm_dir,
        runner="pooling",
        tensor_parallel_size=cfg.llm.tensor_parallel_size,
        max_model_len=cfg.llm.max_model_len,
        gpu_memory_utilization=(
            cfg.search.embeds_gpu_memory_utilization
        ),
        enforce_eager=cfg.llm.enforce_eager,
        distributed_executor_backend=None,
        dtype=cfg.llm.dtype,
        seed=cfg.gen.seed,
    )

prm = build_prm(
    cfg.prm.kind, cfg.prm.prm_dir, device=cfg.prm.device_map,
)
print(f"GPU mem used: {gpu_mem_used_gb():0.2f} GB")

## 4. Pick the question

In [ ]:
load_kwargs = {"ds_split": cfg.data.ds_split}
if cfg.data.level is not None:
    load_kwargs["level"] = cfg.data.level
dataset = load_data_hf(cfg.data.ds_dir, **load_kwargs)

QUESTION_IDX = 0
record = dataset[QUESTION_IDX]
question = record[cfg.data.question_field]

print(f"{len(dataset)} questions (level {cfg.data.level})")
print(f"--- question {QUESTION_IDX} ---")
print(question)
for key in ("answer", "solution"):
    if key in record:
        print(f"--- reference {key} ---")
        print(record[key])
        break

## 5. Trace verbosity

The search cores log their internals on two levels:

- `FATAL` (50) — per-phase **selection** trace: the frontier
  score breakdown per candidate leaf (cnt: q/u/visits; kube/
  kdepth: density; sem: q / diversity bonus), the selected
  node, running `gen_cnt`;
- `ERROR` (40) — additionally the **generation** details:
  current templated text, raw `generate_k_steps` outputs
  (incl. `stop_reasons`), per-candidate PRM scores.

The modules set the root logger *above* FATAL at import (that
is what keeps launchers silent); this cell just lowers it back.
Re-run this cell anytime to change verbosity between runs.

In [ ]:
VERBOSITY = "selection"  # "silent" | "selection" | "generation"

_LEVELS = {
    "silent": logging.FATAL + 1,
    "selection": logging.FATAL,
    "generation": logging.ERROR,
}
root_logger = logging.getLogger()
if not root_logger.handlers:
    logging.basicConfig(format="%(message)s")
root_logger.setLevel(_LEVELS[VERBOSITY])
print(f"trace verbosity: {VERBOSITY}")

## 6. Run the search

Seeding mirrors `_search`'s per-question seed
(`100_000 + trial_idx`), so with the same config this run is
comparable with trial `TRIAL_IDX` of a recorded run.

The trace streams into the cell output AND is written to
`LOG_PATH` (`examine_search_<METHOD>.log`, overwritten each
run of the same method) via a temporary `logging.FileHandler`
on the root logger — handy for a trace too long to scroll
through in the notebook.

Note: at `gen_budget=80` (the recorded-experiments default),
zero completions still happens on a fraction of questions for
**every** BL frontier method — an inherent property of
best-first frontier search, not a bug (measured for
`mcts_bl_cnt_v01` at ~18%, see `docs/findings/coding-findings/`
`bl-cnt-frontier-zero-completion-rate.md`). If you hit it, bump
`QUESTION_IDX` or `gen_budget`.

In [ ]:
TRIAL_IDX = 0
# Method-suffixed so runs of different methods don't overwrite
# each other's traces.
LOG_PATH = f"examine_search_{METHOD}.log"

seed = 100_000 + TRIAL_IDX
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

agent = core_mod.MCTS(config=cfg, question=question)

search_args = [question, agent, cfg, llm_vllm]
if spec["embeds"]:
    search_args.append(llm_vllm_embeds)
search_args.append(prm)

# Tee the trace into LOG_PATH in addition to the cell output, so
# a long trace can be grepped/reviewed outside the notebook.
# Fresh file per run (mode="w") rather than appended across reruns.
file_handler = logging.FileHandler(LOG_PATH, mode="w")
file_handler.setFormatter(logging.Formatter("%(message)s"))
root_logger.addHandler(file_handler)

start = time.time()
try:
    (
        completions, comp_depth, comp_phase, comp_gen,
        q_total_gens, q_last_phase, phase_depths,
        q_nodes_max_depth,
    ) = core_mod.mcts_search(*search_args)
finally:
    root_logger.removeHandler(file_handler)
    file_handler.close()

print(f"\nsearch took {time.time() - start:0.1f}s")
print(f"trace saved to {LOG_PATH}")

## 7. Save the search tree to JSON

Dumps `agent.root` recursively to `TREE_JSON_PATH`
(`examine_search_tree_<METHOD>.json`) — one object per node with
its lineage `tag`, stats (`n`/`q`), flags (`is_terminal`/
`is_completed`), the step text it added on top of its parent, and
its `children`. Handy for diffing tree shape across runs/methods
without re-running the search, or loading into a separate viewer.

For the v02 families this is the clearest way to see eager
terminal handling: terminals carry `n`/`q` from their fold even
though they never sat on the frontier.

In [ ]:
import json

# Method-suffixed so trees from different methods don't
# overwrite each other.
TREE_JSON_PATH = f"examine_search_tree_{METHOD}.json"


def node_to_dict(node):
    """Recursive node -> plain-dict conversion for JSON dumping."""
    step_text = node.state["text"]
    if node.parent is not None:
        step_text = step_text.removeprefix(node.parent.state["text"])
    return {
        "tag": node.tag,
        "depth": node.depth,
        "phase": node.phase,
        "gen_cnt": node.gen_cnt,
        "n": node.visit_count(),
        "q": node.q_value(),
        "is_terminal": node.is_terminal,
        "is_completed": node.is_completed,
        "step_text": step_text,
        "children": [node_to_dict(child) for child in node.children],
    }


tree_dict = node_to_dict(agent.root)
with open(TREE_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(tree_dict, f, indent=1, ensure_ascii=False)
    f.write("\n")
print(f"search tree saved to {TREE_JSON_PATH}")

## 8. Run summary

The same per-question fields a recorded
`generate_...--trial-XXX.jsonl` would hold.

In [ ]:
print(f"generations used   : {q_total_gens}/{cfg.search.gen_budget}")
print(f"last phase reached : {q_last_phase}")
print(f"phase_depths       : {phase_depths}")
print(f"nodes at max depth : {q_nodes_max_depth}")
print(f"completions        : {len(completions)}")
for i, (dep, ph, gc) in enumerate(
    zip(comp_depth, comp_phase, comp_gen)
):
    print(f"  [{i}] depth={dep}  phase={ph}  gen_cnt={gc}")

## 9. Completed nodes

Full text of every EOS/length-terminated leaf (the pool that
`completions` is deduped from), with the tree coordinates and
the node's final q-value.

In [ ]:
if not agent.completed_nodes:
    print("no completed nodes (budget exhausted before any "
          "EOS/length stop)")
for node in agent.completed_nodes:
    text = node.state["text"]
    print(f"=== node {node.tag}  depth={node.depth} "
          f"phase={node.phase} gen_cnt={node.gen_cnt} "
          f"q={node.q_value():0.3f} ===")
    print(text if len(text) <= 1500 else text[:1500] + "\n[...]")
    print()

## 10. Per-step PRM re-score

Re-score one completion step by step — handy for checking how a
node's aggregated score came about. Step/score counts can
differ by one because `PRM.score` splits a bogus trailing empty
step (see `docs/findings/coding-findings/`
`prm-step-split-trailing-separator.md`).

In [ ]:
if completions:
    COMP_IDX = 0
    text = completions[COMP_IDX]
    steps = [s for s in text.split("\n\n") if s.strip()]
    scores = prm.score([question], [[text]])[0][0]
    print(f"{len(steps)} steps, {len(scores)} PRM scores")
    agg = core_mod.aggregate_scores(scores, cfg.gen.agg_strategy)
    print(f"aggregated ({cfg.gen.agg_strategy}): {agg:0.4f}\n")
    print_step_scores(steps, scores)
else:
    print("no completions to score")